In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_parquet('yellow_tripdata_2023-01.parquet')
df = df.sample(n=500000, random_state=42)
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

In [ ]:
df_clean = df[
    (df['trip_duration_minutes'] >= 1) &
    (df['trip_duration_minutes'] <= 180) &
    (df['trip_distance'] > 0) &
    (df['passenger_count'] > 0)
].copy()
print("Clean rows:", len(df_clean))

In [ ]:
zones = pd.read_csv('nyc_taxi_zone_lookup.csv')
zones['LocationID'] = zones['LocationID'].astype('Int64')
df_clean['PULocationID'] = df_clean['PULocationID'].astype('Int64')
df_clean['DOLocationID'] = df_clean['DOLocationID'].astype('Int64')

df_clean = df_clean.merge(zones[['LocationID','borough']], left_on='PULocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'borough':'pickup_borough'}).drop(columns=['LocationID'])

df_clean = df_clean.merge(zones[['LocationID','borough']], left_on='DOLocationID', right_on='LocationID', how='left')
df_clean = df_clean.rename(columns={'borough':'dropoff_borough'}).drop(columns=['LocationID'])